# 02 · Run rMATS: QKI knockdown vs control

Third of six notebooks (00 screen and locus → 01 download → **02 rMATS** → 03 panel c → 04 eCLIP and rbp-maps → 05 optional window maps). Input: the BAMs and `manifest.tsv` from notebook 01.

**What ENCODE ran.** ENCODE quantified splicing with **rMATS v3.2.1.beta** against **GENCODE v19**, comparing the two knockdown replicates (sample 1) with the two non-target control replicates (sample 2). Panel c uses its skipped-exon (SE) results counted from junction reads only (`SE.MATS.JunctionCountOnly.txt`). ENCODE called an event significant when |ΔPSI| > 0.05, P < 0.05 and FDR < 0.05 (Van Nostrand et al. 2020, Methods). ENCODE did not publish the exact command line. With rMATS 3.2.1's required arguments and its defaults, it would be:

```bash
# rMATS 3.2.1.beta (Python 2; shown for reference only)
python RNASeq-MATS.py \
    -b1 QKI_KD_rep1.bam,QKI_KD_rep2.bam \
    -b2 control_rep1.bam,control_rep2.bam \
    -gtf gencode.v19.annotation.gtf \
    -t paired -len 100 \
    -o QKI-LV08-K562
```

**What this notebook runs.** rMATS 3.2.1 needs Python 2 and is no longer maintained, so this notebook uses its successor, **rMATS-turbo (4.x)**, with the same inputs and GENCODE v40 (GRCh37 mapping) instead of v19, once per knockdown (QKI, K562 and HepG2). Its `SE.MATS.JC.txt` has the same 23 columns as the 3.2.1 `SE.MATS.JunctionCountOnly.txt`, so notebooks 03 and 04 accept either. Counts can differ between rMATS versions, and v40 adds and retires some transcripts relative to v19, so a few events appear in only one table, so section 4 compares the rerun with ENCODE's release, for the two PICALM exons and genome-wide.

Runtime: roughly 30-60 min per knockdown with 8 threads; memory about 8 GB.

## Accessing RBP-ARK data through Globus

RBP-ARK keeps every Skipper output on one Globus collection. Public files can be downloaded straight over HTTPS with no login; bulk copies go through Globus.

| | |
| --- | --- |
| Globus collection | **RBP-ARK**, `9cd9be17-cbed-4b34-ab76-704b2fd6ac7b` ([open in the Globus web app](https://app.globus.org/file-manager?origin_id=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b&origin_path=%2F)) |
| HTTPS address of the same files | `https://g-f8db23.0ed28.75bc.data.globus.org/<path>` |

**Where things are.** `<version>/<run>` is the Skipper version and run folder. Since November 2026 every ENCODE3, ENCORE and ENCORE2 dataset on rbp-ark.com (gene search, dataset pages, BDBags) is Skipper `2.0.0/v2`, and so are these notebooks. Earlier runs (`1.99.0/v1`, `1.100.0/v1`) remain on the collection.

| Path on the collection | Contents |
| --- | --- |
| `/encode3/eclip/Skipper/<version>/<run>/output/` | ENCODE3 eCLIP datasets, named `<RBP>_<cell>_ENCSR…` |
| `/encore2/eclip/Skipper/<version>/<run>/output/` | ENCORE2 eCLIP datasets with 5000-series IDs (e.g. `QKI_K562_5007`) |
| `/encore/eclip/Skipper/<version>/<run>/output/` | ENCORE eCLIP datasets with 4000-series IDs |
| `/summary_data/` | summary tables (QC, window counts) |

Inside each `output/` folder, files are grouped by type, then named by dataset:

| Subfolder | Files |
| --- | --- |
| `reproducible_enriched_windows/` | `<dataset>.reproducible_enriched_windows.tsv.gz`, the enriched windows |
| `reproducible_enriched_re/` | enriched repetitive elements |
| `secondary_results/bams/dedup/genome_R2/` | deduplicated, read-2-only genome BAMs (ENCODE3) |
| `secondary_results/bams/dedup/genome/` | deduplicated genome BAMs (ENCORE) |
| `secondary_results/bams/raw/genome/` | alignments before deduplication |
| `secondary_results/bigwigs/scaled/plus/`, `.../minus/` | strand-specific signal tracks |
| `QC/`, `gene_sets/`, `homer/` | QC reports, gene set enrichment, motifs |

**Three ways to get files.**

1. **One dataset: BDBag.** Each dataset page on rbp-ark.com has a **Download BDBag** link. The bag's `fetch.txt` lists every file of that dataset with its HTTPS URL, size and path, and `manifest-sha256.txt` its checksum. Notebook 04 reads the same file list from the RBP-ARK API and downloads only the BAMs it needs.
2. **Many datasets: Globus web app.** Sign in at [app.globus.org](https://app.globus.org) with your institution or a free Globus ID, open the RBP-ARK collection (link above), select folders or files, and transfer them to a collection you can write to: your laptop via [Globus Connect Personal](https://www.globus.org/globus-connect-personal), or your institution's cluster or storage collection.
3. **Scripted: Globus CLI.**

```bash
pip install globus-cli
globus login                                   # once; opens a browser
RBPARK=9cd9be17-cbed-4b34-ab76-704b2fd6ac7b
OUT=/encode3/eclip/Skipper/2.0.0/v2/output

# list what is there
globus ls "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/" | grep QKI

# copy one file to this computer (Globus Connect Personal must be running)
DEST=$(globus endpoint local-id)
globus transfer "$RBPARK:$OUT/secondary_results/bams/dedup/genome_R2/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" \
    "$DEST:$HOME/rbpark/QKI_K562_ENCSR366YOG_IP_1.genome.Aligned.sort.dedup.R2.bam" --label "QKI eCLIP"

# or copy a whole folder
globus transfer --recursive "$RBPARK:$OUT/reproducible_enriched_windows/" "$DEST:$HOME/rbpark/windows/"
globus task list                               # follow progress
```

If the CLI asks for extra consent the first time it touches the collection, it prints the exact `globus session consent ...` command to run. The Globus CLI commands are shown for reference and were not run in testing; the BDBag and HTTPS routes were. Please copy only what you need: the public HTTPS endpoint is shared, and bulk sweeps have taken it offline.

**In this notebook:** nothing comes from RBP-ARK; rMATS runs on the ENCODE knockdown BAMs. RBP-ARK data is used in notebooks 00, 04 and 05.

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam

sys.path.insert(0, ".")
import cs3_utils as U

ROOT, DATA = U.folders()
BAM_DIR = DATA / "downloads" / "shrna_bams_hg19"
REF_DIR = DATA / "downloads" / "encode_reference"
RMATS_DIR = DATA / "rmats"
GTF = REF_DIR / "gencode.v40lift37.annotation.gtf"
GTF_URL = ("https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_40/GRCh37_mapping/"
           "gencode.v40lift37.annotation.gtf.gz")

THREADS = int(os.environ.get("RMATS_THREADS", os.cpu_count() or 4))
READ_LENGTH = 100            # ENCODE shRNA RNA-seq: 2 x 100 nt
LIB_TYPE = "fr-firststrand"  # dUTP stranded libraries (ENCODE: strand specificity "reverse"); see section 3
KNOCKDOWNS = list(U.KNOCKDOWNS)

# Run rmats.py with this kernel's Python so its compiled module matches (realpath keeps sys.path[0] = rMATS dir)
RMATS = [sys.executable, os.path.realpath(shutil.which("rmats.py"))]
RMATS_VERSION = subprocess.run(RMATS + ["--version"], capture_output=True, text=True).stdout.strip()
SETTINGS = "annotation {}\nlibType {}\nrMATS {}".format(GTF.name, LIB_TYPE, RMATS_VERSION)
print(SETTINGS)
print(subprocess.check_output(RMATS + ["--version"], text=True).strip())

manifest = pd.read_csv(BAM_DIR / "manifest.tsv", sep="\t")
manifest

## 1. GENCODE v40 annotation, GRCh37 mapping

The case studies use GENCODE v40 throughout. The BAMs are aligned to hg19, so rMATS needs hg19 coordinates: GENCODE publishes each release mapped back to GRCh37 (`gencode.v40lift37.annotation.gtf`, same transcript models and IDs as v40). ENCODE's own rMATS run used GENCODE v19.

In [ ]:
if not GTF.exists():
    gz = Path(str(GTF) + ".gz")
    subprocess.check_call(["curl", "-L", "-sS", "--retry", "5", "-o", str(gz), GTF_URL])
    subprocess.check_call(["gunzip", "-f", str(gz)])
n_genes = sum(1 for line in open(GTF) if "\tgene\t" in line)
print(GTF.name, "-", n_genes, "genes")

## 2. Check the BAMs

rMATS skips reads whose length differs from `--readLength` unless `--variable-read-length` is set. The cell below samples reads from each BAM to confirm 100-nt paired-end reads, and checks that the chromosomes are `chr`-prefixed (GENCODE v19 uses `chr11`, not `11`).

In [ ]:
def bam_profile(path, n=200_000):
    """Returns a summary of the BAM at `path` from its first `n` reads: contig names, modal read length, the fraction of
    100-nt reads and the fraction paired.
    """
    lengths, paired = [], 0
    with pysam.AlignmentFile(path) as bam:
        refs = list(bam.references)
        for i, read in enumerate(bam.fetch(until_eof=True)):
            if i >= n:
                break
            if read.is_secondary or read.is_supplementary:
                continue
            lengths.append(read.infer_read_length() or read.query_length)
            paired += read.is_paired
    lengths = np.array(lengths)
    return {"n_contigs": len(refs), "n_chr_contigs": sum(r.startswith("chr") for r in refs),
            "has_chr11": "chr11" in refs, "spike_in_contigs": sum(r.startswith("ERCC") for r in refs),
            "modal_read_length": int(np.bincount(lengths).argmax()),
            "fraction_100nt": round(float((lengths == READ_LENGTH).mean()), 3),
            "fraction_paired": round(paired / len(lengths), 3)}


profiles = pd.DataFrame([dict(accession=a, **bam_profile(b)) for a, b in zip(manifest["accession"], manifest["bam"])])
assert (profiles["modal_read_length"] == READ_LENGTH).all()
assert profiles["has_chr11"].all(), "chromosome names must be chr-prefixed, as in GENCODE"   # ERCC spike-in contigs are listed first and are ignored
profiles

## 3. Run rMATS-turbo

| Flag | Value | Why |
| --- | --- | --- |
| `--b1` | the two shRNA knockdown BAMs | sample 1 = knockdown, as in ENCODE's files, so ΔPSI = knockdown − control |
| `--b2` | the two non-target control BAMs | sample 2 = control |
| `--gtf` | GENCODE v40 (GRCh37 mapping) | the annotation used throughout; ENCODE used v19 |
| `-t paired --readLength 100` | | 2 × 100 nt reads; sets IncFormLen and SkipFormLen |
| `--variable-read-length` | | keeps any trimmed reads instead of silently dropping them |
| `--libType` | `fr-firststrand` | the libraries are dUTP strand-specific (ENCODE lists strand specificity "reverse"), so only reads on the event's strand are counted. ENCODE's 3.2.1 run does not record its setting |
| `--nthread` | all cores | |

Novel splice sites are off (the default), as in 3.2.1. A finished run is detected and skipped; it is redone automatically if the annotation, `--libType` or the rMATS version changes (recorded in `run_settings.txt`). Delete `<data>/rmats/rerun/<knockdown>/` to force a rerun.

In [ ]:
def run_rmats(key):
    """Runs rMATS-turbo for knockdown `key` (2 knockdown vs 2 control BAMs) into <data>/rmats/rerun/<key>/, unless a
    finished run with the same settings is there.
    """
    sub = manifest[manifest["knockdown"] == key].sort_values("rep")
    kd = sub.loc[sub["condition"] == "knockdown", "bam"].tolist()
    ctl = sub.loc[sub["condition"] == "control", "bam"].tolist()
    assert len(kd) == 2 and len(ctl) == 2, sub

    od, tmp = RMATS_DIR / "rerun" / key, RMATS_DIR / "tmp" / key
    od.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)   # leftovers from an interrupted run make rMATS count BAMs twice
    tmp.mkdir(parents=True, exist_ok=True)
    (od / "b1.txt").write_text(",".join(kd) + "\n")
    (od / "b2.txt").write_text(",".join(ctl) + "\n")
    result = od / "SE.MATS.JC.txt"
    stamp = od / "run_settings.txt"
    if result.exists() and not (stamp.exists() and stamp.read_text().strip() == SETTINGS):
        print(key, "- earlier run used other settings; rerunning with", SETTINGS.replace("\n", "; "))
        for f in od.iterdir():
            if f.name not in ("b1.txt", "b2.txt"):
                shutil.rmtree(f) if f.is_dir() else f.unlink()
    if result.exists():
        print(key, "- already done:", result)
        return result

    cmd = RMATS + [
        "--b1", str(od / "b1.txt"), "--b2", str(od / "b2.txt"),
        "--gtf", str(GTF), "-t", "paired", "--readLength", str(READ_LENGTH),
        "--variable-read-length", "--libType", LIB_TYPE,
        "--nthread", str(THREADS), "--od", str(od), "--tmp", str(tmp),
    ]
    (od / "command.txt").write_text(" ".join(cmd) + "\n")
    print(" ".join(cmd))
    start = time.time()
    with open(od / "rmats.log", "w") as log:
        status = subprocess.call(cmd, stdout=log, stderr=subprocess.STDOUT)
    print(key, "- exit", status, "after {:.0f} min".format((time.time() - start) / 60))
    print("".join(open(od / "rmats.log").readlines()[-5:]))
    assert status == 0 and result.exists(), "rMATS failed; see " + str(od / "rmats.log")
    stamp.write_text(SETTINGS + "\n")
    return result


rerun_files = {key: run_rmats(key) for key in KNOCKDOWNS}

## 4. Compare with ENCODE's release

First the two PICALM exons. hg19, minus strand, 0-based starts:

| Exon | Exon | Upstream exon (lower coordinate) | Downstream exon |
| --- | --- | --- | --- |
| exon 13 (150 nt) | `chr11:85,701,292-85,701,442` | `85,694,908-85,695,016` | `85,707,868-85,707,972` |
| 24-nt exon | `chr11:85,689,112-85,689,136` | `85,687,665-85,687,725` | `85,692,171-85,692,271` |

Expected from ENCODE (exon 13): ΔPSI +0.22 (QKI K562), +0.48 (QKI HepG2). 24-nt exon: −0.12 (QKI K562), −0.53 (QKI HepG2).

In [ ]:
def read_se(path):
    """Returns the rMATS skipped-exon table at `path` (3.2.1 SE.MATS.JunctionCountOnly.txt or rMATS-turbo
    SE.MATS.JC.txt), with a `key` column of its coordinates.
    """
    df = pd.read_csv(path, sep="\t")
    for col in ("GeneID", "geneSymbol"):
        df[col] = df[col].astype(str).str.strip('"')
    df["key"] = df[U.COORDS].astype(str).agg(":".join, axis=1)
    return df


def extract_encode_se(key):
    """Returns ENCODE's released SE.MATS.JunctionCountOnly.txt for knockdown `key`, extracted once from its rMATS tar
    into <data>/rmats/encode_release/<key>/.
    """
    acc = U.KNOCKDOWNS[key]["rmats"]
    out = RMATS_DIR / "encode_release" / key / "SE.MATS.JunctionCountOnly.txt"
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(REF_DIR / (acc + ".tar.gz")) as tar:
            member = next(m for m in tar.getmembers() if m.name.endswith("MATS_output/SE.MATS.JunctionCountOnly.txt"))
            out.write_bytes(tar.extractfile(member).read())
    return out


def event_row(df, exon):
    """Returns the row of rMATS table `df` for PICALM `exon` (hg19 coordinates), or None unless exactly one row matches.
    """
    hit = df
    for k, v in U.PICALM_HG19[exon].items():
        hit = hit[hit[k].astype(str) == str(v)]
    return hit.iloc[0] if len(hit) == 1 else None


tables, rows = {}, []
for key in KNOCKDOWNS:
    tables[key] = {"rerun": read_se(rerun_files[key]), "encode": read_se(extract_encode_se(key))}
    for source, df in tables[key].items():
        for exon in U.PICALM_HG19:
            r = event_row(df, exon)
            if r is None:
                continue
            rows.append({"knockdown": key, "exon": exon, "source": source,
                         "IJC_KD": r["IJC_SAMPLE_1"], "SJC_KD": r["SJC_SAMPLE_1"],
                         "IJC_control": r["IJC_SAMPLE_2"], "SJC_control": r["SJC_SAMPLE_2"],
                         "PSI_KD": r["IncLevel1"], "PSI_control": r["IncLevel2"],
                         "dPSI": r["IncLevelDifference"], "FDR": r["FDR"]})
pd.DataFrame(rows).sort_values(["exon", "knockdown", "source"])

In [ ]:
def significant(df):
    """Returns the rows of rMATS table `df` with FDR < 0.05, P < 0.05 and |dPSI| > 0.05."""
    return df[(df["FDR"] < 0.05) & (df["PValue"] < 0.05) & (df["IncLevelDifference"].abs() > 0.05)]


ex13_key = ":".join(str(v) for v in U.PICALM_HG19["exon13"].values())
fig, axes = plt.subplots(1, len(KNOCKDOWNS), figsize=(4.2 * len(KNOCKDOWNS), 4.2))
for ax, key in zip(np.atleast_1d(axes), KNOCKDOWNS):
    rerun, encode = tables[key]["rerun"], tables[key]["encode"]
    shared = rerun.merge(encode, on="key", suffixes=("_rerun", "_encode"))
    sig_r, sig_e = set(significant(rerun)["key"]), set(significant(encode)["key"])
    r = np.corrcoef(shared["IncLevelDifference_rerun"], shared["IncLevelDifference_encode"])[0, 1]
    print("{}: {:,} events rerun, {:,} ENCODE, {:,} shared (dPSI r = {:.3f}); significant: {:,} rerun, {:,} ENCODE, {:,} in both"
          .format(key, len(rerun), len(encode), len(shared), r, len(sig_r), len(sig_e), len(sig_r & sig_e)))
    ax.scatter(shared["IncLevelDifference_encode"], shared["IncLevelDifference_rerun"], s=2, alpha=0.3, color="#52514e", lw=0)
    g = shared[shared["key"] == ex13_key]
    ax.scatter(g["IncLevelDifference_encode"], g["IncLevelDifference_rerun"], s=40, color="#eb6834", label="PICALM exon 13")
    ax.plot([-1, 1], [-1, 1], color="#b4b3ad", lw=0.8, ls="--")
    ax.set(title="{} (r = {:.3f})".format(key.replace("_", " "), r), xlabel="ΔPSI, ENCODE release (rMATS 3.2.1)",
           ylabel="ΔPSI, this rerun (rMATS-turbo)", xlim=(-1, 1), ylim=(-1, 1))
    ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

Notebooks 03 and 04 read `<data>/rmats/rerun/<knockdown>/SE.MATS.JC.txt` by default (`SE_SOURCE=rerun`), or ENCODE's `<data>/rmats/encode_release/<knockdown>/SE.MATS.JunctionCountOnly.txt` with `SE_SOURCE=encode`. The draft figure used ENCODE's release.